In [ ]:
def get_processable_files(src):
    """
    Returns a list of processable files from the given path.
    """
    ##############################################
    # Imports
    ##############################################
    from dotenv import load_dotenv
    import os
    load_dotenv()
    
    files = [f for f in os.listdir(src) if ".pdf" in f]

    return files

In [ ]:
def get_chapter_ranges(sourcefilename, do_print=True):
    """
    Returns a list of (beginPage, endPage) ranges for chunks that represent chapters in the given pdf.
    """

    ##############################################
    # Imports
    ##############################################
    from dotenv import load_dotenv
    import os
    load_dotenv()
    import pypdfium2 as pdfium
    
    print("Getting chapter ranges...\n")
    
    pdf = pdfium.PdfDocument(sourcefilename)
    
    ranges = []
    
    begin, end = None, None
    
    for item in pdf.get_toc():
        
        state = "*" if item.n_kids == 0 else "-" if item.is_closed else "+"
        
        target = "?" if item.page_index is None else item.page_index+1
        
        boundary = None
        
        if item.page_index and ((item.n_kids == 0 and item.level < 2) or item.level == 2):
            
            if begin is not None:
                
                end = item.page_index - 1
                
                boundary = [begin, max(begin, end)]
                
                ranges.append(boundary)
                
            begin = item.page_index
            
        if do_print:
            
            if boundary:
                
                print("    " * 2 +  f"(Pages {(boundary[0]+1)} - {(boundary[1]+1)})" + "\n")
                
            print(("    " * item.level) + f"[{state}] {item.title} -> {target}  # {item.view_mode} {item.view_pos}")
            
    return ranges

In [ ]:
def split_chapters(sourcefilename, targetfilename, pagerange):
    """
    Splits the pdf into chapters using the provided page ranges.
    Returns the name of the new pdf chunk.
    """

    ##############################################
    # Imports
    ##############################################
    from dotenv import load_dotenv
    import os
    load_dotenv()
    import pypdfium2 as pdfium
    from pathlib import Path
    
    try:
        
        source_pdf = pdfium.PdfDocument(sourcefilename)
        
        new_pdf = pdfium.PdfDocument.new()
    
        print(f"Retrieving chapter...{targetfilename}, Pages {pagerange[0]} to {pagerange[1]}")
        
        new_page_index = new_pdf.import_pages(source_pdf, pages=list(range(pagerange[0], pagerange[1]+1)))
        
        new_pdf.save(targetfilename)
        
        source_pdf.close()
        
        new_pdf.close()
        
    except Exception as e:
        
        print(f"Error saving {targetfilename}: {e}")

In [ ]:
def convert_to_markdown(pdffile, markdownfile):
    """
    Converts the pdf into a markdown file.
    """
    
    ##############################################
    # Imports
    ##############################################
    from dotenv import load_dotenv
    import os
    load_dotenv()
    from docling.document_converter import DocumentConverter
    
    try:
        print(f"Converting {pdffile} to markdown...")
        
        converter = DocumentConverter()
        
        result = converter.convert(pdffile)
        
        markdown_output = result.document.export_to_markdown()

        with open(markdownfile, "w") as file:
            
            file.write(markdown_output)

        print(f"{markdownfile} generated.")
        
    except Exception as e:
        print(f"Error saving {markdownfile}: {e}")
    

In [ ]:
def generate_markdown_section_raw_data(file):
    """
    Generates markdown section chunks from the file.
    """

    ##############################################
    # Imports
    ##############################################
    from datasets import Dataset, Features, Value
    from langchain_text_splitters import RecursiveCharacterTextSplitter, MarkdownHeaderTextSplitter
    from langchain.docstore.document import Document
    from sdg_hub.core.blocks import PromptBuilderBlock, LLMChatBlock, LLMParserBlock
    from datasets import Dataset, concatenate_datasets
    import traceback
    import re
    import uuid
    import pprint

    dataset = None

    def strip_code_section(content):
        """
        Strips out code sections of file.
        """
        code_sections = re.findall(r'([^`]+)```([^`]+)```', content, re.DOTALL | re.MULTILINE)
        
        return code_sections
    
    try:
        print(f"Parsing markdown {file}...")
        
        filecontent = None
        
        with open(file, mode="r") as f: 
            
            filecontent = f.read()

            if strip_code_section(filecontent):

                print(f"Starting code-to-text mappings for {file}...")
                
                headers_to_split = [("#", "Header 1"), ("##", "Header 2"),("###", "Header 3")]
                
                text_splitter = MarkdownHeaderTextSplitter(headers_to_split, strip_headers=False)
            
                splits = text_splitter.split_text(filecontent)
        
                sections = [[strip_code_section(split.page_content) for split in splits if split]][0]

                sections = [(str(uuid.uuid4()), section) for section in sections if section]
                
                dataset = Dataset.from_list([{"code_id": section_id, "code": c, "markdown": s} 
                                              for section_id, section in sections for s, c in section])

        return dataset        

    except Exception as e:

        print(f"Error occurred while parsing markdown {file}: {e}")

        traceback.print_exc()

In [ ]:
# ##############################################
# # Imports
# ##############################################
from dotenv import load_dotenv
import os
load_dotenv()
from pathlib import Path
from datasets import Dataset, concatenate_datasets

source_path = 'pdf_source'

target_path_chapters = 'pdf_chunked_target'

target_path_markdown = 'pdf_chunked_markdown'

target_path_jsonl = "json"

for directory_path in [source_path, 
                       
                       target_path_chapters, 
                       
                       target_path_markdown,
                      
                       target_path_jsonl]:
        
    Path(directory_path).mkdir(parents=True, exist_ok=True)

files = get_processable_files(source_path)

dataset = None

for file in files:
    
    ranges = get_chapter_ranges(f"{source_path}/{file}", do_print=False)
    
    for idx, _range in enumerate(ranges):
        
        pdf = f"{target_path_chapters}/{idx}_{file}"
        
        md = f"{target_path_markdown}/{idx}_{file.replace('.pdf', '.md')}"
        
        split_chapters(f"{source_path}/{file}", pdf, _range)
        
        convert_to_markdown(pdf, md)

        dataset = generate_markdown_section_raw_data(md) if not dataset else concatenate_datasets([dataset, generate_markdown_section_raw_data(md)])

print("Writing raw dataset to jsonl file...")

dataset.to_json(f"{target_path_jsonl}/data.jsonl")

In [ ]:
##############################################
# sdg_hub
##############################################

from datasets import load_dataset, DatasetDict
from sdg_hub.core.flow import FlowRegistry, Flow
import nest_asyncio
nest_asyncio.apply()

flow_path = "flows/graphrag_knowledge_generation/flow.yaml"

columns_to_keep = ["code_id", "code", "markdown", "summary", "summary_type", "eval_summary_relevance", "eval_summary_faithfulness"]

flow = Flow.from_yaml(flow_path)

flow.set_model_config(
    model="openrouter/openai/gpt-oss-20b",
    api_base=f"{os.getenv('OPENROUTER_API_BASE')}",
    api_key=os.getenv("OPENROUTER_TOKEN"),
)

datasets_config = {}

for split in ["train"]:

    dataset = load_dataset("json", data_files=f"{target_path_jsonl}/data.jsonl", split=split)
    
    converted_dataset = flow.generate(dataset)
    
    columns_to_remove = [col for col in converted_dataset.column_names if col not in columns_to_keep]
    
    source_dataset = converted_dataset.remove_columns(columns_to_remove)

    source_dataset.to_json(f"{target_path_jsonl}/source_data_{split}.jsonl")

    datasets_config[split] = source_dataset

final_dataset = DatasetDict(datasets_config)

final_dataset.push_to_hub("oaawofolu/emerson")